# Cognee + Claude Agent SDK Integration Guide

This notebook demonstrates how to integrate **Cognee** (a semantic memory for AI agents) with **Claude Agent SDK** to create AI agents with persistent, cross-session memory capabilities powered by knowledge graphs + embeddings.

## What is Cognee?

**Cognee** is an open-source semantic memory layer that transforms unstructured, structured, semi-structured data into queryable knowledge graphs backed by embeddings. Cognee:

- **Automatically extracts** entities, relationships, and semantic meaning from text
- **Creates knowledge graphs** with embeddings 
- **Enables natural language queries** to retrieve relevant information
- **Maintains context** across different sessions and interactions
- **Supports multi-modal data** including text, documents, and structured data

## What is Claude Agent SDK? 

**Claude Agent SDK** is Anthropic's framework for building AI agents with Claude. It provides:

- **ClaudeSDKClient**: Easy-to-use client for interacting with Claude
- **Tool Integration**: Seamless integration with custom tools via MCP servers
- **Message Types**: Structured message types (UserMessage, AssistantMessage, etc.)
- **Streaming Support**: Real-time response streaming

## Why Combine Them?

The Cognee-Claude integration provides AI agents with persistent, semantic memory that survives between sessions. Agents can store information in Cognee's knowledge graph and retrieve it using natural language queries, enabling more accurate retrieval, continuity across conversations without manual state management.

Let's explore how this works in practice!


## 📋 Prerequisites

Before running this notebook, make sure you have the required dependencies installed:

```bash
# Using uv
uv sync

# Or using pip
pip install cognee-integration-claude
```


## Environment Setup

Both Claude Agent SDK and Cognee require API keys for LLM operations. Let's configure the environment:


In [ ]:
import os

# Set your API keys here or use environment variables
# Get your OpenAI API key from: https://platform.openai.com/api-keys
# Check cognee docs to use other LLMs: https://docs.cognee.ai/setup-configuration/llm-providers

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY")

# The Claude Agent SDK uses a bundled CLI that handles authentication automatically.
# - In Cursor / Claude Code: Uses your existing Cursor/Claude session
# - Standalone: First run guides you through OAuth or API key authentication

## Claude Agent SDK Fundamentals

Before diving into the Cognee integration, let's understand Claude Agent SDK's core building blocks:

### ClaudeSDKClient
**ClaudeSDKClient** is the central abstraction in Claude Agent SDK. It encapsulates:
- Connection to Claude via Anthropic's API
- MCP server configuration for tools
- Tool permissions (allowed/disallowed)

### ClaudeAgentOptions
**ClaudeAgentOptions** configures the client:
- **mcp_servers**: Dictionary of MCP servers providing tools
- **allowed_tools**: Tools the agent can use
- **disallowed_tools**: Tools to disable

### Tools
**Tools** are functions that agents can call. Claude Agent SDK uses:
- **@tool decorator**: For creating async tool functions
- **create_sdk_mcp_server**: For bundling tools into MCP servers

### Message Types
The SDK provides structured message types:
- **UserMessage**: Messages from the user
- **AssistantMessage**: Responses from Claude
- **SystemMessage**: System-level messages
- **ResultMessage**: Final result messages with metadata

Let's create a simple setup to understand the basics:


In [6]:
from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ClaudeSDKClient,
    ResultMessage,
    SystemMessage,
    TextBlock,
    UserMessage,
    create_sdk_mcp_server,
)


def display_message(msg):
    """Standardized message display function.

    - UserMessage: "User: <content>"
    - AssistantMessage: "Claude: <content>"
    - SystemMessage: ignored
    - ResultMessage: "Result ended" + cost if available
    """
    if isinstance(msg, UserMessage):
        for block in msg.content:
            if isinstance(block, TextBlock):
                print(f"User: {block.text}")
    elif isinstance(msg, AssistantMessage):
        for block in msg.content:
            if isinstance(block, TextBlock):
                print(f"Claude: {block.text}")
    elif isinstance(msg, SystemMessage):
        # Ignore system messages
        pass
    elif isinstance(msg, ResultMessage):
        print("Result ended")


print("✓ Helper functions defined")

✓ Helper functions defined


## Introducing Cognee to Claude: Semantic Memory for Agents

Now let's add **persistent semantic memory** to our agents using Cognee!

### What Cognee Adds to Claude:

- **Semantic Memory**: Store and retrieve information using natural language from Cognee's knowledge graph backed by embeddings
- **Knowledge Graphs**: Automatic entity and relationship extraction
- **Cross-Session Persistence**: Memory survives between different agent instances
- **Intelligent Search**: Find relevant information by meaning using Cognee's advanced retrieval methods
- **Session Isolation**: Keep different users' data separate and secure

### Cognee Tools Integration

The integration provides two main tools:
- **`add_tool`**: Store information in Cognee's knowledge graph
- **`search_tool`**: Query stored information using natural language

Let's start fresh by pruning any existing Cognee data:


In [7]:
import cognee
from cognee.api.v1.config import config

# Configure Cognee data directories (optional - uses defaults if not set)
config.data_root_directory(os.path.join(os.getcwd(), "../.cognee/data_storage"))
config.system_root_directory(os.path.join(os.getcwd(), "../.cognee/system"))

# Clean slate - remove any existing data
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)

print("✓ Cognee data pruned successfully!")


2025-12-16T17:19:42.519866 [info     ] Deleted Kuzu database files at /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/cognee_integration_claude/../.cognee/system/databases/cognee_graph_kuzu [cognee.shared.logging_utils]

2025-12-16T17:19:44.527942 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 


✓ Cognee data pruned successfully!


### Importing Cognee Tools

Now let's import the Cognee tools and create an agent with semantic memory capabilities:


In [8]:
from cognee_integration_claude import add_tool, search_tool

# Create an MCP server with Cognee tools
cognee_server = create_sdk_mcp_server(
    name="cognee-tools", version="1.0.0", tools=[add_tool, search_tool]
)

# Configure the agent with Cognee memory tools
memory_options = ClaudeAgentOptions(
    mcp_servers={"tools": cognee_server},
    allowed_tools=["mcp__tools__add_tool", "mcp__tools__search_tool"],
    disallowed_tools=[
        "Task",
        "Bash",
        "Glob",
        "Grep",
        "ExitPlanMode",
        "Read",
        "Edit",
        "Write",
        "NotebookEdit",
        "WebFetch",
        "TodoWrite",
        "WebSearch",
        "BashOutput",
        "KillShell",
        "SlashCommand",
    ],
)

print("✓ Memory agent configured with Cognee tools: [add_tool, search_tool]")

✓ Memory agent configured with Cognee tools: [add_tool, search_tool]


### Storing Information with Cognee

Let's give our agent some contract information to remember. The agent will use the `add_tool` to store this in Cognee's knowledge graph:


In [9]:
contracts_data = """
We have signed a contract with the following company: "Meditech Solutions". Company is in the healthcare industry. Start date is Jan 2023 and end date is Dec 2025. Contract value is £1.2M.
We have signed a contract with the following company: "QuantumSoft". Company is in the technology industry. Start date is Aug 2024 and end date is Aug 2028. Contract value is £5.5M.
We have signed a contract with the following company: "Orion Retail Group". Company is in the retail industry. Start date is Mar 2024 and end date is Mar 2026. Contract value is £850K.
"""

print("=== ADDING CONTRACTS TO MEMORY ===")
async with ClaudeSDKClient(options=memory_options) as client:
    await client.query(f"Please remember this contract information: {contracts_data}")

    async for msg in client.receive_response():
        display_message(msg)

print("\n✓ Contract data sent to Cognee memory!")


Using bundled Claude Code CLI: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/.venv/lib/python3.13/site-packages/claude_agent_sdk/_bundled/claude


=== ADDING CONTRACTS TO MEMORY ===



add_tool called with: "Contract with Meditech Solutions:
- Company: Meditech Solutions
- Industry: Healthcare
- Start Date: January 2023
- End Date: December 2025
- Contract Value: £1.2M"

Adding data to cognee: Contract with Meditech Solutions:
- Company: Meditech Solutions
- Industry: Healthcare
- Start Date: January 2023
- End Date: December 2025
- Contract Value: £1.2M


User a0ca8399-05f5-4d01-a75c-171ca35850ff has registered.



2025-12-16T17:19:56.275347 [info     ] Pipeline run started: `ab8f54f8-98b9-5fd7-91d1-07f07964be39` [run_tasks_with_telemetry()]

2025-12-16T17:19:56.275871 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:19:56.276328 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2025-12-16T17:19:56.292275 [info     ] Registered loader: pypdf_loader [cognee.infrastructure.loaders.LoaderEngine]

2025-12-16T17:19:56.292891 [info     ] Registered loader: text_loader [cognee.infrastructure.loaders.LoaderEngine]

2025-12-16T17:19:56.293274 [info     ] Registered loader: image_loader [cognee.infrastructure.loaders.LoaderEngine]

2025-12-16T17:19:56.293564 [info     ] Registered loader: audio_loader [cognee.infrastructure.loaders.LoaderEngine]

2025-12-16T17:19:56.293933 [info     ] Registered loader: csv_loader  [cognee.infrastructure.loaders.LoaderEngine]

2025-12-16T17:19:56.294176 [info     ] Registered loader: unstructured_loader [cogn

Claude: Perfect! I've stored all three contracts in my knowledge base. Here's a summary of what I've saved:

1. **Meditech Solutions** (Healthcare)
   - Duration: Jan 2023 - Dec 2025
   - Value: £1.2M

2. **QuantumSoft** (Technology)
   - Duration: Aug 2024 - Aug 2028
   - Value: £5.5M

3. **Orion Retail Group** (Retail)
   - Duration: Mar 2024 - Mar 2026
   - Value: £850K

I can retrieve this information whenever you need it. Just ask me about any of these contracts or companies!
Result ended

✓ Contract data sent to Cognee memory!


### Background Data Ingestion

Let's also add some additional data directly to Cognee (bypassing the agent) to demonstrate how the agent can access pre-existing knowledge. This simulates a scenario where:
- **Historical data** exists in the knowledge base
- **Agent interactions** add new information
- **Both sources** are searchable together


In [10]:
# Add data directly to Cognee (simulating pre-existing knowledge base)
additional_contracts = """
We have signed a contract with the following company: "HealthBridge Systems". Company is in the healthcare industry. Start date is Feb 2023 and end date is Jan 2026. Contract value is £2.4M.
We have signed a contract with the following company: "NovaCare Diagnostics". Company is in the healthcare industry. Start date is May 2024 and end date is Apr 2027. Contract value is £1.6M.
We have signed a contract with the following company: "SkyPort Logistics". Company is in the logistics industry. Start date is Nov 2022 and end date is Oct 2026. Contract value is £3.1M.
"""

print("Adding background data directly to Cognee...")
await cognee.add(additional_contracts)
await cognee.cognify()
print("✓ Background data processed and added to knowledge graph!")


2025-12-16T17:21:59.609307 [info     ] Pipeline run started: `ab8f54f8-98b9-5fd7-91d1-07f07964be39` [run_tasks_with_telemetry()]

2025-12-16T17:21:59.609643 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:21:59.609985 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2025-12-16T17:21:59.625829 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]


Adding background data directly to Cognee...



2025-12-16T17:21:59.626243 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:21:59.626480 [info     ] Pipeline run completed: `ab8f54f8-98b9-5fd7-91d1-07f07964be39` [run_tasks_with_telemetry()]

2025-12-16T17:21:59.631889 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2025-12-16T17:21:59.681609 [info     ] Pipeline run started: `0509b3ec-150b-5184-82fe-2fd5bd02ecf1` [run_tasks_with_telemetry()]

2025-12-16T17:21:59.682245 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2025-12-16T17:21:59.683258 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2025-12-16T17:21:59.690756 [info     ] Coroutine task started: `extract_graph_from_data` [run_tasks_base]

2025-12-16T17:22:49.926326 [info     ] No close match found for 'company' in category 'classes' [OntologyAdapter]

2025-12-16T17:22:49.928146 [info     ] No close matc

✓ Background data processed and added to knowledge graph!


### Visualizing the Knowledge Graph

Let's visualize what Cognee has built from our data:


In [11]:
import webbrowser


async def visualize_graph(file_name, open_browser=True):
    destination_file_path = os.path.join(os.getcwd(), file_name)
    await cognee.visualize_graph(destination_file_path)

    if open_browser:
        url = "file://" + os.path.abspath(destination_file_path)
        webbrowser.open(url)

    print(f"✓ Graph visualization saved to: {destination_file_path}")


await visualize_graph(file_name="first_graph_visualization.html")


2025-12-16T17:23:03.930322 [info     ] Retrieved 52 nodes and 104 edges in 0.01 seconds [cognee.shared.logging_utils]

2025-12-16T17:23:04.071091 [info     ] Graph visualization saved as /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/cognee_integration_claude/first_graph_visualization.html [cognee.shared.logging_utils]

2025-12-16T17:23:04.071696 [info     ] The HTML file has been stored at path: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/cognee_integration_claude/first_graph_visualization.html [cognee.shared.logging_utils]


✓ Graph visualization saved to: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/cognee_integration_claude/first_graph_visualization.html


Open the generated HTML file in your browser to explore the knowledge graph. You'll see:
- Entities extracted from the contracts (companies, industries, values)
- Relationships between entities
- Data from both agent interactions and background ingestion


## Cross-Session Memory Persistence

Now let's demonstrate one of Cognee's key features: **persistent memory across agent instances**.

We'll create a completely fresh agent instance that:
- Has **no conversation history** from the previous agent
- Has **no internal state** carried over
- **CAN access** all information stored in Cognee's knowledge graph

This shows how Cognee provides **true persistent memory** that survives agent restarts:


In [12]:
# Create a FRESH agent instance - no shared state with previous agent
search_query = "I need to research our contract portfolio. Can you search for any contracts we have with companies in the healthcare industry?"

print("=== SEARCHING WITH FRESH AGENT ===")
print(f"Query: {search_query}\n")

async with ClaudeSDKClient(options=memory_options) as fresh_client:
    await fresh_client.query(search_query)

    print("=== AGENT RESPONSE ===")
    async for msg in fresh_client.receive_response():
        display_message(msg)


Using bundled Claude Code CLI: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/.venv/lib/python3.13/site-packages/claude_agent_sdk/_bundled/claude


=== SEARCHING WITH FRESH AGENT ===
Query: I need to research our contract portfolio. Can you search for any contracts we have with companies in the healthcare industry?

=== AGENT RESPONSE ===
Claude: I'll search the knowledge base for contracts with healthcare industry companies.



search_tool called with: "contracts healthcare industry companies"

Searching cognee for: contracts healthcare industry companies with node_set: None

2025-12-16T17:23:11.708243 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.07s [cognee.shared.logging_utils]

2025-12-16T17:23:11.708792 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2025-12-16T17:23:11.714669 [info     ] ID-filtered retrieval: 52 nodes and 104 edges in 0.01s [cognee.shared.logging_utils]

2025-12-16T17:23:11.715476 [info     ] Graph projection completed: 52 nodes, 104 edges in 0.00s [CogneeGraph]

Search results: ['Contracts with healthcare companies:\n- HealthBridge Systems — Feb 2023 to Jan 2026 — £2.4M\n- NovaCare Diagnostics — May 2024 to Apr 2027 — £1.6M\n- Meditech Solutions — Jan 2023 to Dec 2025 — £1.2M\nTotal value: £5.2M']


Claude: Great! I found information about your healthcare industry contracts. Here's what's in your contract portfolio:

## Healthcare Industry Contracts

1. **HealthBridge Systems**
   - Duration: February 2023 to January 2026
   - Value: £2.4M

2. **NovaCare Diagnostics**
   - Duration: May 2024 to April 2027
   - Value: £1.6M

3. **Meditech Solutions**
   - Duration: January 2023 to December 2025
   - Value: £1.2M

**Total Portfolio Value: £5.2M**

You have three active contracts with healthcare companies, with a combined value of £5.2 million. The HealthBridge Systems contract is your largest healthcare contract, and NovaCare Diagnostics has the longest remaining term (through April 2027).
Result ended


**Amazing!** The fresh agent found all the healthcare contracts, including:
- Contracts added through agent interactions (Meditech Solutions)
- Contracts added through background ingestion (HealthBridge Systems, NovaCare Diagnostics)

This demonstrates true **cross-session persistence** - the knowledge survives agent restarts!


## Sessions with Cognee

In multi-user applications, you often need to **isolate data between users** while still maintaining persistent memory for each user. Cognee supports this through **session-based tools**.

### How Session Isolation Works

The `get_sessionized_cognee_tools()` function creates tools that are bound to a specific session ID:

```python
def get_sessionized_cognee_tools(session_id: Optional[str] = None) -> list:
    """
    Returns a list of cognee tools sessionized for a specific user.
    
    Args:
        session_id: The session ID to bind to all tools.
                    If None, a UUID-based ID is auto-generated.
    
    Returns:
        list: [sessionized_add_tool, sessionized_search_tool]
    """
```

When you use sessionized tools:
- Data added is tagged with the session ID
- Searches only return data from that session
- Different sessions are completely isolated


In [13]:
# Start fresh for the session demo
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✓ Starting fresh for session demonstration")


2025-12-16T17:23:23.855272 [info     ] Deleted Kuzu database files at /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/cognee_integration_claude/../.cognee/system/databases/cognee_graph_kuzu [cognee.shared.logging_utils]

2025-12-16T17:23:25.904277 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 


✓ Starting fresh for session demonstration


### Creating Session-Isolated Agents

Let's create two agents for two different users, each with their own isolated memory:


In [14]:
from cognee_integration_claude import get_sessionized_cognee_tools

# Create sessionized tools for User A (Alice)
alice_add_tool, alice_search_tool = get_sessionized_cognee_tools("alice-session")

alice_server = create_sdk_mcp_server(
    name="alice-tools", version="1.0.0", tools=[alice_add_tool, alice_search_tool]
)

alice_options = ClaudeAgentOptions(
    mcp_servers={"tools": alice_server},
    allowed_tools=["mcp__tools__add_tool", "mcp__tools__search_tool"],
    disallowed_tools=[
        "Task",
        "Bash",
        "Glob",
        "Grep",
        "ExitPlanMode",
        "Read",
        "Edit",
        "Write",
        "NotebookEdit",
        "WebFetch",
        "TodoWrite",
        "WebSearch",
        "BashOutput",
        "KillShell",
        "SlashCommand",
    ],
)

# Create sessionized tools for User B (Bob)
bob_add_tool, bob_search_tool = get_sessionized_cognee_tools("bob-session")

bob_server = create_sdk_mcp_server(
    name="bob-tools", version="1.0.0", tools=[bob_add_tool, bob_search_tool]
)

bob_options = ClaudeAgentOptions(
    mcp_servers={"tools": bob_server},
    allowed_tools=["mcp__tools__add_tool", "mcp__tools__search_tool"],
    disallowed_tools=[
        "Task",
        "Bash",
        "Glob",
        "Grep",
        "ExitPlanMode",
        "Read",
        "Edit",
        "Write",
        "NotebookEdit",
        "WebFetch",
        "TodoWrite",
        "WebSearch",
        "BashOutput",
        "KillShell",
        "SlashCommand",
    ],
)

print("✓ Created isolated agents for Alice and Bob")


Initialized session with session_id = alice-session

Initialized session with session_id = bob-session


✓ Created isolated agents for Alice and Bob


### Adding Data to Different Sessions

Let's add different contracts to each user's session:


In [15]:
# Alice's contracts (Insurance industry)
alice_contracts = [
    'Contract with "Guardian Insurance Ltd" - Insurance industry, Feb 2023 to Feb 2026, £1.8M',
    'Contract with "Pioneer Assurance Group" - Insurance industry, Oct 2024 to Oct 2029, £4.2M',
]

print("=== ALICE'S SESSION ===")
for contract in alice_contracts:
    print(f"Adding: {contract[:50]}...")
    async with ClaudeSDKClient(options=alice_options) as alice_client:
        await alice_client.query(f"Remember this: {contract}")
        async for msg in alice_client.receive_response():
            if isinstance(msg, AssistantMessage):
                for block in msg.content:
                    if isinstance(block, TextBlock):
                        print(
                            f"  Alice's agent: {block.text[:80]}..."
                            if len(block.text) > 80
                            else f"  Alice's agent: {block.text}"
                        )

# Bob's contracts (Technology industry)
bob_contracts = [
    'Contract with "TechNova Solutions" - Technology industry, Jan 2024 to Jan 2027, £3.5M',
    'Contract with "DataStream Analytics" - Technology industry, Mar 2024 to Mar 2028, £2.1M',
]

print("\n=== BOB'S SESSION ===")
for contract in bob_contracts:
    print(f"Adding: {contract[:50]}...")
    async with ClaudeSDKClient(options=bob_options) as bob_client:
        await bob_client.query(f"Remember this: {contract}")
        async for msg in bob_client.receive_response():
            if isinstance(msg, AssistantMessage):
                for block in msg.content:
                    if isinstance(block, TextBlock):
                        print(
                            f"  Bob's agent: {block.text[:80]}..."
                            if len(block.text) > 80
                            else f"  Bob's agent: {block.text}"
                        )

print("\n✓ Contracts added to separate sessions!")


Using bundled Claude Code CLI: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/.venv/lib/python3.13/site-packages/claude_agent_sdk/_bundled/claude


=== ALICE'S SESSION ===
Adding: Contract with "Guardian Insurance Ltd" - Insurance...



sessionized add_tool called with: "Contract with "Guardian Insurance Ltd" - Insurance industry, Feb 2023 to Feb 2026, £1.8M"

Using tool add_tool_impl with user_id: alice-session

Adding data to cognee: Contract with "Guardian Insurance Ltd" - Insurance industry, Feb 2023 to Feb 2026, £1.8M

2025-12-16T17:23:31.150122 [info     ] Pipeline run started: `e9fa0c65-6a32-53b1-ae18-6a752778ceed` [run_tasks_with_telemetry()]

2025-12-16T17:23:31.150495 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:23:31.150873 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2025-12-16T17:23:31.166859 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2025-12-16T17:23:31.167231 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:23:31.167589 [info     ] Pipeline run completed: `e9fa0c65-6a32-53b1-ae18-6a752778ceed` [run_tasks_with_telemetry()]


User 5e436472-eb4c-4583-bdb7-cebecf8c01c6 has registered.



2025-12-16T17:23:33.173722 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2025-12-16T17:23:33.206299 [info     ] Pipeline run started: `c5381f3c-3302-5ea9-ad2c-bbf8002ee270` [run_tasks_with_telemetry()]

2025-12-16T17:23:33.206690 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2025-12-16T17:23:33.207318 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2025-12-16T17:23:33.212802 [info     ] Coroutine task started: `extract_graph_from_data` [run_tasks_base]

2025-12-16T17:24:05.295789 [info     ] Reconnecting to Kuzu database... [cognee.shared.logging_utils]

2025-12-16T17:24:05.369325 [info     ] Loaded JSON extension          [cognee.shared.logging_utils]

2025-12-16T17:24:05.382994 [info     ] No close match found for 'company' in category 'classes' [OntologyAdapter]

2025-12-16T17:24:05.383585 [info     ] No close match found for 'guardian insurance ltd' i

...

  Alice's agent: I've stored that information about the Pioneer Assurance Group contract. The det...

=== BOB'S SESSION ===
Adding: Contract with "TechNova Solutions" - Technology in...



sessionized add_tool called with: "Contract with "TechNova Solutions" - Technology industry, Jan 2024 to Jan 2027, £3.5M"

Using tool add_tool_impl with user_id: bob-session

Adding data to cognee: Contract with "TechNova Solutions" - Technology industry, Jan 2024 to Jan 2027, £3.5M

2025-12-16T17:25:25.660859 [info     ] Pipeline run started: `e9fa0c65-6a32-53b1-ae18-6a752778ceed` [run_tasks_with_telemetry()]

2025-12-16T17:25:25.661241 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:25:25.661652 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2025-12-16T17:25:25.677853 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2025-12-16T17:25:25.678182 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:25:25.678418 [info     ] Pipeline run completed: `e9fa0c65-6a32-53b1-ae18-6a752778ceed` [run_tasks_with_telemetry()]

2025-12-16T17:25:27.684392 [info     ] No 

  Bob's agent: I've stored the information about the TechNova Solutions contract in my knowledg...
Adding: Contract with "DataStream Analytics" - Technology ...



sessionized add_tool called with: "Contract with "DataStream Analytics" - Technology industry, Mar 2024 to Mar 2028, £2.1M"

Using tool add_tool_impl with user_id: bob-session

Adding data to cognee: Contract with "DataStream Analytics" - Technology industry, Mar 2024 to Mar 2028, £2.1M

2025-12-16T17:26:14.789392 [info     ] Pipeline run started: `e9fa0c65-6a32-53b1-ae18-6a752778ceed` [run_tasks_with_telemetry()]

2025-12-16T17:26:14.789751 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:26:14.790246 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2025-12-16T17:26:14.808331 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2025-12-16T17:26:14.808694 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2025-12-16T17:26:14.808979 [info     ] Pipeline run completed: `e9fa0c65-6a32-53b1-ae18-6a752778ceed` [run_tasks_with_telemetry()]

2025-12-16T17:26:16.815982 [info     ]

  Bob's agent: I've stored the contract information for DataStream Analytics:
- **Company**: Da...

✓ Contracts added to separate sessions!


### Demonstrating Session Isolation

Now let's verify that each user can only see their own data:


In [16]:
# Alice searches for contracts
print("=== ALICE SEARCHING ===")
async with ClaudeSDKClient(options=alice_options) as alice_client:
    await alice_client.query("What contracts do we have? Search for all contracts.")

    print("Alice's results:")
    async for msg in alice_client.receive_response():
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, TextBlock):
                    print(block.text)

print("\n" + "=" * 50 + "\n")

# Bob searches for contracts
print("=== BOB SEARCHING ===")
async with ClaudeSDKClient(options=bob_options) as bob_client:
    await bob_client.query("What contracts do we have? Search for all contracts.")

    print("Bob's results:")
    async for msg in bob_client.receive_response():
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, TextBlock):
                    print(block.text)


Using bundled Claude Code CLI: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/.venv/lib/python3.13/site-packages/claude_agent_sdk/_bundled/claude


=== ALICE SEARCHING ===
Alice's results:
I'll search the knowledge base for information about contracts.



sessionized search_tool called with: "contracts"

Using tool search_tool_impl with user_id: alice-session

Searching cognee for: contracts with node_set: ['alice-session']

2025-12-16T17:27:07.346776 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.07s [cognee.shared.logging_utils]

2025-12-16T17:27:07.347420 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]

2025-12-16T17:27:07.360776 [info     ] Graph projection completed: 17 nodes, 82 edges in 0.00s [CogneeGraph]

Search results: ['Two contracts found:\n1) Contract with Guardian Insurance Ltd — Insurance industry; Feb 2023 to Feb 2026; value £1.8M.\n2) Contract with Pioneer Assurance Group — Insurance industry; Oct 2024 to Oct 2029; value £4.2M.']

Using bundled Claude Code CLI: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/.venv/lib/python3.13/site-packages/claude_agent_sdk/_bundled/claude


Based on the search results, you have **2 contracts**:

1. **Guardian Insurance Ltd**
   - Industry: Insurance
   - Duration: February 2023 to February 2026
   - Value: £1.8M

2. **Pioneer Assurance Group**
   - Industry: Insurance
   - Duration: October 2024 to October 2029
   - Value: £4.2M

Both contracts are in the insurance industry, with a combined value of £6M.


=== BOB SEARCHING ===
Bob's results:
I'll search the knowledge base for information about contracts.



sessionized search_tool called with: "contracts"

Using tool search_tool_impl with user_id: bob-session

Searching cognee for: contracts with node_set: ['bob-session']

2025-12-16T17:27:20.053037 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.07s [cognee.shared.logging_utils]

2025-12-16T17:27:20.053761 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]

2025-12-16T17:27:20.066082 [info     ] Graph projection completed: 17 nodes, 80 edges in 0.00s [CogneeGraph]

Search results: ['Two contracts found:\n1) Contract with DataStream Analytics — Technology industry; Mar 2024 to Mar 2028; £2.1M.\n2) Contract with TechNova Solutions — Technology industry; Jan 2024 to Jan 2027; £3.5M.']


Based on the search results, you have **two contracts**:

1. **Contract with DataStream Analytics**
   - Industry: Technology
   - Duration: March 2024 to March 2028 (4 years)
   - Value: £2.1M

2. **Contract with TechNova Solutions**
   - Industry: Technology
   - Duration: January 2024 to January 2027 (3 years)
   - Value: £3.5M

Both contracts are in the technology industry with a combined total value of £5.6M.


**Notice:** 
- Alice only sees her insurance contracts
- Bob only sees his technology contracts
- Complete session isolation is maintained!


### Visualizing Session Clusters

Let's visualize the knowledge graph to see how sessions create distinct clusters:


In [17]:
await visualize_graph(file_name="sessionized_graph_visualization.html")


2025-12-16T17:27:26.945991 [info     ] Retrieved 45 nodes and 109 edges in 0.00 seconds [cognee.shared.logging_utils]

2025-12-16T17:27:26.948184 [info     ] Graph visualization saved as /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/cognee_integration_claude/sessionized_graph_visualization.html [cognee.shared.logging_utils]

2025-12-16T17:27:26.948768 [info     ] The HTML file has been stored at path: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/cognee_integration_claude/sessionized_graph_visualization.html [cognee.shared.logging_utils]


✓ Graph visualization saved to: /Users/handekafkas/Documents/local-code/integrations/cognee-integration-claude/cognee_integration_claude/sessionized_graph_visualization.html


In the visualization, you'll observe **session-based data clustering**:

1. **Alice's Cluster**: Data from `alice-session` grouped together
2. **Bob's Cluster**: Data from `bob-session` in a separate cluster
3. **Global Data**: Information added outside sessions forms its own cluster

This demonstrates how Cognee maintains both **session isolation** and **global knowledge accessibility**.


## Understanding Session ID Generation

Let's explore how session management works under the hood.

### Automatic Session IDs

If you don't provide a session ID, one is auto-generated:

```python
# Auto-generated session ID
add_tool, search_tool = get_sessionized_cognee_tools()
# Results in: cognee-test-user-<uuid>
```

### Custom Session IDs

For production applications, you'll want to use meaningful session IDs based on:
- User authentication tokens
- Organization IDs
- Custom identifiers from your system


In [18]:
# Example: Using authentication-based session IDs
async def get_user_session_id() -> str:
    """Simulate getting user ID from authentication system"""
    # In production, this would come from your auth system
    return "user-12345-authenticated"


# Create tools with auth-based session ID
user_session_id = await get_user_session_id()
auth_add_tool, auth_search_tool = get_sessionized_cognee_tools(user_session_id)

print(f"✓ Created tools with custom session ID: {user_session_id}")


Initialized session with session_id = user-12345-authenticated


✓ Created tools with custom session ID: user-12345-authenticated


## Summary

In this guide, we've learned how to:

### Claude Agent SDK Fundamentals
- ✅ Configure agents with `ClaudeAgentOptions`
- ✅ Create tools using the `@tool` decorator
- ✅ Bundle tools into MCP servers with `create_sdk_mcp_server`
- ✅ Execute agents using `ClaudeSDKClient`

### Cognee Integration
- ✅ Use `add_tool` to store information in the knowledge graph
- ✅ Use `search_tool` to query stored information
- ✅ Demonstrate cross-session memory persistence
- ✅ Visualize knowledge graphs

### Session Management
- ✅ Create session-isolated tools with `get_sessionized_cognee_tools()`
- ✅ Maintain data isolation between users
- ✅ Use custom session IDs for production applications

### Key Takeaways

1. **Cognee provides persistent memory** that survives agent restarts
2. **Session isolation** keeps user data separate and secure
3. **Knowledge graphs** enable semantic search across stored information
4. **Easy integration** with Claude Agent SDK through simple tool imports

## Next Steps

- Explore advanced Cognee features like custom pipelines
- Integrate with other data sources (documents, PDFs)
- Deploy to production
- Check out the [Cognee documentation](https://docs.cognee.ai) for more capabilities
